# Lab: Automate Test Case Generation from Jira Stories using Azure OpenAI

In this lab you will build a small **agent** that:

1. Connects to a Jira project and fetches a user story by key.  
2. Extracts requirement‑style information from the story (summary, description, acceptance criteria).  
3. Uses an OpenAI/Azure OpenAI to generate structured test cases.  

> ⚠️ **Note:** This notebook is meant as a starting point. You will still need valid Jira credentials and a Jira Cloud project.


## 1. Architecture Overview

High‑level flow:

1. **Config & Secrets** – Configure Jira URL, user/email, and API token (kept outside the code when possible).  
2. **Jira Connector (Tool)** – A Python function `fetch_jira_issue(issue_key)` that calls the Jira REST API.  
3. **Requirement Extractor** – Function that pulls `summary`, `description`, and (optionally) acceptance criteria from the Jira issue JSON.  
4. **LLM Wrapper** – Azure OpenAI/OpenAI
5. **Test Generation Agent** – Orchestrator that:
   - fetches the story →
   - extracts requirements →
   - builds a prompt →
   - calls the LLM →
   - returns test cases in a Markdown table.


## Story details

**Summary**  
Add order filter panel to Orders page

**Issue Type**  
Story  

**Description**  

**User Story**  
As a customer support agent, I want to filter the list of customer orders by date range and order status so that I can quickly find the relevant orders while handling customer queries.

**Business Context**  
Support agents often receive calls and chats from customers asking about specific orders. Currently, agents must scroll through long lists of orders or manually search by order ID. This is time-consuming and error-prone. A clear filter panel will help agents quickly narrow down results and improve average handling time.

---

### Functional Requirements

1. **Filter Panel Placement**
   - The Orders page must display a filter panel **above** the orders table.
   - The filter panel must be visible by default (no collapsed state initially).

2. **Order Status Filter**
   - The filter panel must include an **Order Status** dropdown.
   - The Order Status filter must support the following options:
     - Pending  
     - Processing  
     - Shipped  
     - Delivered  
     - Cancelled  
   - The default selection should be **All statuses** (no filter applied).

3. **Date Range Filter**
   - The filter panel must include a **Date Range** filter with:
     - Start Date (From)
     - End Date (To)
   - If only Start Date is selected, results must include orders with `order_date >= Start Date`.
   - If only End Date is selected, results must include orders with `order_date <= End Date`.
   - If both dates are selected, results must include orders with `Start Date <= order_date <= End Date`.

4. **Filter Application Behavior**
   - The Orders table must refresh when the user clicks an **"Apply Filters"** button.
   - Filtered results must reflect both:
     - Selected Order Status (if any), and  
     - Selected Date Range (if any).
   - If no filters are set, the Orders table must show the default list (e.g., all orders from the last 30 days).

5. **Clear / Reset Filters**
   - The filter panel must include a **"Clear Filters"** button.
   - Clicking "Clear Filters" must:
     - Reset status to **All statuses**.
     - Clear any selected dates.
     - Refresh the Orders table back to the default state.

---

### Non-Functional Requirements

1. Filtering must complete and render updated results within **2 seconds** for up to **10,000 orders**.
2. Filters must be applied on the **backend** (server-side or API level), not only in the UI.
3. The filter panel and orders table must be fully responsive on screens down to **1024px** width.

---

### Acceptance Criteria

1. **Status Filter – Single Status**
   - Given I am on the Orders page  
   - And there are orders with statuses Pending, Processing, Shipped, Delivered, and Cancelled  
   - When I select **“Shipped”** in the Order Status filter and click **Apply Filters**  
   - Then the Orders table must show **only orders with status = Shipped**.

2. **Status Filter – All Statuses**
   - Given I am on the Orders page  
   - When I select **“All statuses”** and click **Apply Filters**  
   - Then the Orders table must show orders of **any status** (no status filter applied).

3. **Date Range Filter – Both Dates**
   - Given I am on the Orders page  
   - And there are orders placed between `2025-01-01` and `2025-01-31`  
   - When I set the Start Date to `2025-01-10` and the End Date to `2025-01-20` and click **Apply Filters**  
   - Then the Orders table must show **only orders where `order_date` is between 10 Jan and 20 Jan (inclusive)**.

4. **Date Range + Status Combined**
   - Given I am on the Orders page  
   - And there are Pending and Shipped orders for the last 30 days  
   - When I set the Start Date to 7 days ago, End Date to today, select **“Pending”** status, and click **Apply Filters**  
   - Then the Orders table must show **only Pending orders placed in the last 7 days**.

5. **Clear Filters**
   - Given I have applied any combination of status and date filters  
   - When I click **Clear Filters**  
   - Then the Order Status must reset to **All statuses**  
   - And both Start Date and End Date must be cleared  
   - And the Orders table must revert to the **default unfiltered view**.


## 2. Environment Setup

Run the cell below **once** to install dependencies (uncomment if needed).  
If you're using a managed environment that already has `transformers` and `torch`, you can skip the install.


In [ ]:
# If you're in a fresh environment (e.g., Colab), uncomment and run:
!pip install -q transformers torch requests
!pip install openai azure-identity python-dotenv


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.3/191.3 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.3/213.3 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 7.3 MB/s eta 0:00:00


## 3. Imports


In [ ]:
import os
import json
import getpass
from typing import Dict, Any

import requests
import torch
from transformers import pipeline


## 4. Jira Configuration

We will read Jira configuration from environment variables to avoid hard‑coding secrets.

Required values:

- `JIRA_BASE_URL` – e.g. `https://your-domain.atlassian.net`
- `JIRA_EMAIL` – your Jira login email / username
- `JIRA_API_TOKEN` – Jira API token (you can generate one from your Atlassian account)

You can either:

- Set them in your shell _before_ starting Jupyter, or  
- Input them interactively in this notebook (not saved to file).

### Access API Token and create a story
#### API Token
- Go to https://id.atlassian.com/manage-profile/security/api-tokens -> Create API Token -> Create -> Copy the API token

#### Create a story
- With an existing Jira account log in to https://your-domain.atlassian.net (e.g. https://meteoros-team-jhc5bzmb.atlassian.net/)

- Click on Create adjacent to search bar -> Type "Story" -> Enter Summary and Description and save

- Note down the story key (e.g. KAN-4)



In [ ]:
# --- Jira configuration ---

os.environ["JIRA_BASE_URL"] = "https://meteoros-team-jhc5bzmb.atlassian.net/"  # TODO: replace
os.environ["JIRA_EMAIL"] = "pratham@meteoros.in"                # TODO: replace
os.environ["JIRA_API_TOKEN"] = ""          # TODO: replace


# Ask for API token if it's not already set as an env var
if not os.environ.get("JIRA_API_TOKEN"):
    print("JIRA_API_TOKEN is not set in environment. It will be requested interactively.")
    os.environ["JIRA_API_TOKEN"] = getpass.getpass("Enter your Jira API token (input hidden): ")

JIRA_API_TOKEN = os.environ["JIRA_API_TOKEN"]



In [ ]:
import os
from openai import AzureOpenAI
os.environ["AZURE_OPENAI_ENDPOINT"] = ""
os.environ["AZURE_OPENAI_API_KEY"] = ""
os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"] = "gpt-4o"
# Read env vars
endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
api_key = os.environ["AZURE_OPENAI_API_KEY"]
deployment_name = os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"]  # this is your *deployment* name

# Create Azure OpenAI client
client = AzureOpenAI(
    api_key=api_key,
    api_version="2025-01-01-preview",          # check docs for latest version
    azure_endpoint=endpoint,
)



## 5. Jira Connector "Tool"

A minimal helper that calls the Jira REST API and returns the issue JSON.  
We are using the Jira Cloud v3 API: `/rest/api/3/issue/{issueKey}`.


In [ ]:
def fetch_jira_issue(issue_key: str) -> Dict[str, Any]:
    """
    Fetch a Jira issue by key (e.g. "PROJ-123") and return the JSON response.
    """
    url = f"{os.environ.get("JIRA_BASE_URL")}rest/api/3/issue/{issue_key}"
    auth = (os.environ.get("JIRA_EMAIL"), JIRA_API_TOKEN)

    response = requests.get(url, auth=auth)
    try:
        response.raise_for_status()
    except requests.HTTPError as e:
        print("Error while calling Jira:", e)
        print("Response text:", response.text[:1000])
        raise

    return response.json()


# Quick smoke test (update ISSUE_KEY to a real story in your project)
TEST_ISSUE_KEY = "KAN-5"  # TODO: replace with a real Jira key, e.g. "KAN-4"
try:
    test_issue = fetch_jira_issue(TEST_ISSUE_KEY)
    print(f"Fetched issue {TEST_ISSUE_KEY}. Issue type:", test_issue.get("fields", {}).get("issuetype", {}).get("name"))
except Exception as e:
    print("Test fetch failed (this is expected until you provide a real issue key).")
    print("Error:", e)


Fetched issue KAN-5. Issue type: Story


## 6. Requirement Extractor

Jira fields are flexible and can include rich‑text content. In this lab we will:

- Always extract `summary`
- Convert `description` to a string
- Optionally extract **acceptance criteria** if your project uses a custom field for it

In [ ]:
def normalize_field(field_value) -> str:
    """
    Convert a Jira field (which may be rich text / nested JSON) into a plain string.
    For simplicity, we just JSON-dump non-string fields.
    """
    if field_value is None:
        return ""
    if isinstance(field_value, str):
        return field_value
    # For rich-text / structured fields, a simple JSON dump is fine for the prompt.
    return json.dumps(field_value, indent=2)


def extract_requirements(issue_json: Dict[str, Any]) -> Dict[str, str]:
    fields = issue_json.get("fields", {})

    summary = fields.get("summary", "")
    description_raw = fields.get("description")


    return {
        "summary": normalize_field(summary),
        "description": normalize_field(description_raw),
    }


# Example (will only work after TEST_ISSUE_KEY is set to a real issue)
if 'test_issue' in globals():
    reqs_preview = extract_requirements(test_issue)
    print("Summary:\n", reqs_preview["summary"], "\n")
    print("Description (truncated):\n", reqs_preview["description"][:500], "\n")


Summary:
 Add order filter panel to Orders page 

Description (truncated):
 {
  "type": "doc",
  "version": 1,
  "content": [
    {
      "type": "paragraph",
      "content": [
        {
          "type": "text",
          "text": "Description",
          "marks": [
            {
              "type": "strong"
            }
          ]
        }
      ]
    },
    {
      "type": "paragraph",
      "content": [
        {
          "type": "text",
          "text": "User Story",
          "marks": [
            {
              "type": "strong"
            }
          ]
 



#### Extract Functional, Non-functional and Acceptance criteria from JIRA story description JSON

In [ ]:
response = client.chat.completions.create(
    model=deployment_name,             # IMPORTANT: this is the deployment name, not raw model ID
    messages=[
        {"role": "system", "content": "Your task is to take the provided JSON data of a Jira Story and extract the functional requirements, non-functional requirements and acceptance criteria"},
        {"role": "user", "content": reqs_preview["description"]},
    ],
    max_tokens=1024
)
full_text = response.choices[0].message.content

In [ ]:
print(full_text)

Based on the provided JSON data of the Jira Story, here are the extracted functional requirements, non-functional requirements, and acceptance criteria:

### Functional Requirements
1. **Filter Panel Placement**
   - The Orders page must display a filter panel **above** the orders table.
   - The filter panel must be visible by default (no collapsed state initially).

2. **Order Status Filter**
   - The filter panel must include an **Order Status** dropdown.
   - The Order Status filter must support the following options:
     - Pending
     - Processing
     - Shipped
     - Delivered
     - Cancelled
   - The default selection should be **All statuses** (no filter applied).

3. **Date Range Filter**
   - The filter panel must include a **Date Range** filter with:
     - Start Date (From)
     - End Date (To)
   - If only Start Date is selected, results must include orders with `order_date >= Start Date`.
   - If only End Date is selected, results must include orders with `order_date 

## 7. Prompt Setup

We will use GPT-4o-mini from Azure OpenAI for this lab


In [ ]:
system_prompt = """
You are a QA Engineer.

Your task is to create comprehensive, high-quality test cases for the Jira story described below.

Use ALL of the following inputs:
- Functional Requirements
- Non-Functional Requirements
- Acceptance Criteria

Follow these guidelines:

1. Cover:
   - Happy path / positive scenarios
   - Negative scenarios (validation, error handling, invalid inputs)
   - Edge and boundary cases (date ranges, empty filters, large datasets, etc.)
2. Ensure each test case clearly maps back to at least one requirement or acceptance criterion.
3. Avoid repeating identical test cases; prefer fewer, higher-quality tests with clear coverage.
4. Use concise, professional wording suitable for a real test management tool.
"""

## 8. Prompt Engineering for Test Generation

We will instruct the model to act as a **senior QA engineer** and to output test cases in a Markdown table.

Columns:

- `ID`
- `Title`
- `Type (Positive/Negative/Edge)`
- `Pre-conditions`
- `Steps`
- `Expected Result`
- `Mapped Requirement(s)`
- `Priority`


In [ ]:
TESTGEN_PROMPT = """
Story Requirements (functional, non-functional, and acceptance criteria):
{requirements_text}

---

Now, based on the above, produce a set of test cases in a Markdown table with the following columns:

| ID | Title | Type (Positive/Negative/Edge) | Pre-conditions | Steps | Expected Result | Mapped Requirement(s) | Priority |

Number the IDs as TC-001, TC-002, and so on. Make sure the table is complete and directly grounded in the given requirements and acceptance criteria.
"""


def build_testcase_prompt(requirements_text: str) -> str:
    return TESTGEN_PROMPT.format(requirements_text=requirements_text)

## 9. Putting It Together: The Test Generation Agent

The "agent" here is a simple orchestrator that:

1. Fetches the Jira story.  
2. Extracts requirement fields.  
3. Builds the LLM prompt.  
4. Calls the model and returns the generated test cases.


In [ ]:
def generate_with_llm(prompt: str, max_new_tokens: int = 512) -> str:


    response = client.chat.completions.create(
        model=deployment_name,             # IMPORTANT: this is the deployment name, not raw model ID
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],

    )
    full_text = response.choices[0].message.content
    return full_text





In [ ]:
class JiraTestGenerationAgent:
    def __init__(self, issue_key: str):
        self.issue_key = issue_key

    def run(self) -> str:
        # Step 1: Fetch Jira story
        issue_json = fetch_jira_issue(self.issue_key)
        prompt = build_testcase_prompt(full_text)

        print("====== Prompt sent to LLM (truncated) ======")
        print(prompt[:1000], "...\n")

        # Step 4: Call LLM
        print("====== Generated Test Cases ======\n")
        output = generate_with_llm(prompt)
        print(output)
        return output


## 10. Run the Agent on a Real Jira Story

Set `ISSUE_KEY` to a real Jira issue (e.g., `KAN-4`, `PROJ-101`, etc.), then run the cell.


In [ ]:
ISSUE_KEY = "KAN-5"
agent = JiraTestGenerationAgent(issue_key=ISSUE_KEY)
testcases_markdown = agent.run()


====== Prompt sent to LLM (truncated) ======

Story Requirements (functional, non-functional, and acceptance criteria):
Based on the provided JSON data of the Jira Story, here are the extracted functional requirements, non-functional requirements, and acceptance criteria:

### Functional Requirements
1. **Filter Panel Placement**
   - The Orders page must display a filter panel **above** the orders table.
   - The filter panel must be visible by default (no collapsed state initially).

2. **Order Status Filter**
   - The filter panel must include an **Order Status** dropdown.
   - The Order Status filter must support the following options:
     - Pending
     - Processing
     - Shipped
     - Delivered
     - Cancelled
   - The default selection should be **All statuses** (no filter applied).

3. **Date Range Filter**
   - The filter panel must include a **Date Range** filter with:
     - Start Date (From)
     - End Date (To)
   - If only Start Date is selected, results must include 

In [ ]:
import csv
import json

# ---------- 1. Parse Markdown table into rows ----------

lines = [line.strip() for line in testcases_markdown.splitlines()]

# Keep only the lines that look like table rows
table_lines = [
    line for line in lines
    if "|" in line and not line.lstrip().startswith("|---")
]

if not table_lines:
    raise ValueError("No markdown table lines found in testcases_markdown")

# First row = header
header_cells = [h.strip() for h in table_lines[0].split("|") if h.strip()]
headers = header_cells

rows = []
for row_line in table_lines[1:]:
    # Skip separator or empty-ish lines
    if set(row_line.replace("|", "").replace("-", "").strip()) == set():
        continue

    cells = [c.strip() for c in row_line.split("|") if c.strip()]
    # Pad / trim to match header length
    if len(cells) < len(headers):
        cells += [""] * (len(headers) - len(cells))
    elif len(cells) > len(headers):
        cells = cells[: len(headers)]

    row_dict = dict(zip(headers, cells))
    rows.append(row_dict)

print(f"Parsed {len(rows)} test cases with columns: {headers}")

# ---------- 2. Save as CSV ----------

csv_filename = f"{ISSUE_KEY}_testcases.csv"

with open(csv_filename, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=headers)
    writer.writeheader()
    writer.writerows(rows)

print(f"✅ CSV saved to {csv_filename}")

# ---------- 3. Save as JSON ----------

json_filename = f"{ISSUE_KEY}_testcases.json"

with open(json_filename, "w", encoding="utf-8") as jsonfile:
    json.dump(rows, jsonfile, ensure_ascii=False, indent=2)

print(f"✅ JSON saved to {json_filename}")


Parsed 13 test cases with columns: ['ID', 'Title', 'Type', 'Pre-conditions', 'Steps', 'Expected Result', 'Mapped Requirement(s)', 'Priority']
✅ CSV saved to KAN-5_testcases.csv
✅ JSON saved to KAN-5_testcases.json


## 11. Extensions & Next Steps

Ideas for extending this lab:

- **Code‑Change Driven Tests**  
  Instead of (or in addition to) Jira stories, parse Git diffs and feed changed functions/classes to the LLM to suggest tests.

- **Save Results Back to Jira**  
  Use the Jira REST API to post the generated test cases as a comment or attach them as a file to the story.

- **Stronger Parsing**  
  Parse Jira rich‑text descriptions and structured acceptance criteria fields more carefully instead of simple JSON dumps.

- **Multi‑Agent Setup**  
  Split responsibilities into multiple agents:
  - _Requirements Agent_ – cleans & normalizes story text.
  - _Test Designer Agent_ – proposes tests.
  - _Reviewer Agent_ – checks coverage and suggests missing cases.
